# 💳 Stripe Export Cleaner & FinOps Transformer
## Clean, Normalize, and Calculate Net Revenue from Raw Stripe Exports

**Pack ID:** `pack_0005` · **Category:** `finops` · **Difficulty:** `beginner` · **Version:** `1.1.0`

---
Standardizes raw Stripe export CSVs into clean accounting ledger data with net revenue, fees, and refund flags without needing API keys.


In [ ]:
import pandas as pd
from pathlib import Path

INPUT_CSV = Path('data/stripe_export.csv')
OUT_CLEANED = Path('outputs/stripe_cleaned.csv')
OUT_CLEANED.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_CSV)
print(f'Ingested {len(df)} raw Stripe transaction rows.')


In [ ]:
# Clean & Transform
df_clean = df.copy()
df_clean['gross_amount_usd'] = round(df_clean['amount_cents'] / 100.0, 2)
df_clean['fee_amount_usd'] = round(df_clean['fee_cents'] / 100.0, 2)
df_clean['refund_amount_usd'] = round(df_clean['refunded_cents'] / 100.0, 2)
df_clean['net_revenue_usd'] = round(
    df_clean['gross_amount_usd'] - df_clean['fee_amount_usd'] - df_clean['refund_amount_usd'], 2
)
df_clean['is_successful'] = df_clean['status'] == 'paid'
df_clean['is_refunded'] = df_clean['refund_amount_usd'] > 0

# Reorder and save
clean_cols = [
    'id', 'created_utc', 'customer_email', 'status', 'is_successful',
    'is_refunded', 'gross_amount_usd', 'fee_amount_usd', 'refund_amount_usd',
    'net_revenue_usd', 'description'
]
df_clean[clean_cols].to_csv(OUT_CLEANED, index=False, encoding='utf-8')
print(f'✅ Written: {OUT_CLEANED}')

# Summary Metrics
total_gross = df_clean[df_clean['is_successful']]['gross_amount_usd'].sum()
total_fees = df_clean['fee_amount_usd'].sum()
total_refunds = df_clean['refund_amount_usd'].sum()
total_net = df_clean[df_clean['is_successful']]['net_revenue_usd'].sum()

print(f'\n💰 FinOps Summary:')
print(f'   Gross Volume: ${total_gross:,.2f}')
print(f'   Stripe Fees:  ${total_fees:,.2f} ({total_fees/total_gross*100:.1f}%)')
print(f'   Refunds:      ${total_refunds:,.2f}')
print(f'   Net Revenue:  ${total_net:,.2f}')


In [ ]:
# Verification
assert OUT_CLEANED.exists() and OUT_CLEANED.stat().st_size > 0
print('✅ Pack 0005 execution completed successfully.')
